# 01 · Ingesta Bronze nacional — revisión 4.0
Ejecute en el primary EMR, kernel American Tower /opt/american-tower/venv/bin/python, después de reiniciar. Se conserva Python, Spark YARN y Sedona ya verificados. La notebook crea su sesión y lee la configuración de 00. Solicita tres executors de dos cores y ejecuta trabajo de selección y transferencia en ellos.

**Correcciones:** colección única `sentinel-2-l2a` para consultar los 29 trimestres 2019Q1–2026Q1; cobertura espacial prioritaria antes de nubosidad; comparación de candidatas/seleccionadas; páginas STAC completas agrupadas como JSONL.gz por trimestre; validación temprana, HEAD concurrentes y plan duradero reanudable. No consulta exclusivamente C1 ni mezcla silenciosamente colecciones. Se conservan el baseline y los offsets publicados en cada Item para la normalización posterior en Silver.

Bronze conserva archivos publicados por proveedores, sin recortar ni remuestrear. Sentinel incluye cuatro bandas por escena y metadatos, hasta una adquisición por tesela y mes. No es un SAFE completo. Operations conserva AOI derivado, selección, cobertura, plan y evidencias. No modifica Silver, Gold ni Terraform.

Se mantiene alcance nacional y umbral de huella de 00. El límite operativo de esta revisión es **8192 GiB (8 TiB)**: una cota, no volumen garantizado. Se muestra el tamaño exacto del plan antes de transferir. Ejecute sección 9 únicamente después de revisar ese output. El trabajo es distribuido en executors con multipart y un archivo simultáneo por host; Spark valida el entorno y manifiesto. No se promete disponibilidad de todos los assets ni ausencia de errores externos.

Complete o compruebe las dos rutas OSIPTEL de sección 1 y sus fechas reales. No use un plan de la revisión anterior: no contenía bandas Sentinel. Los objetos de evidencia de la corrida antigua permanecen; no se borran automáticamente.

**Corrección 3.1 de assets:** las cuatro bandas deben tener HTTPS antes de participar en selección. Metadatos S3 del dataset se resuelven mediante el proxy público documentado; productInfo identifica la ruta real del XML de producto. Se conservan el href original y la evidencia de resolución. Requester Pays permanece deshabilitado. Se reutilizan los mismos catálogos cacheados v3.

**Ejecución distribuida 4.0:** consultas STAC concurrentes por trimestre en driver; filtrado y selección geométrica Spark mapPartitions por trimestre en executors; transferencias Spark con un payload simultáneo por host, recibos individuales y SHA-256 remoto. Cache de catálogo v3 compatible; nuevo cache de selección v4. El número de hosts efectivos se mide, no se presupone.


## 0. Sesión Spark y Python del clúster
Ejecute esta celda primero después del reinicio. Si hay un SparkContext activo, detiene con una instrucción explícita. No instale PySpark desde pip. La prueba SQL debe mostrar un POINT y los workers el Python de /opt. Descargar los JAR exige acceso HTTPS a Maven Central.

In [ ]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

PYTHON_ENV = "/opt/american-tower/venv/bin/python"
SPARK_HOME = Path("/usr/lib/spark")

assert sys.executable == PYTHON_ENV, (
    f"Selecciona el kernel {PYTHON_ENV}. Actual: {sys.executable}"
)
assert version("apache-sedona") == "1.7.2"

# Seleccionar Python antes de crear cualquier SparkContext.
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["PYSPARK_PYTHON"] = PYTHON_ENV
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_ENV

py4j = sorted(
    (SPARK_HOME / "python" / "lib").glob("py4j-*.zip")
)
if not py4j:
    raise RuntimeError("No se encontró Py4J en la instalación de EMR.")

for path in [SPARK_HOME / "python", py4j[0]]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pyspark import SparkContext
from sedona.spark import SedonaContext

if SparkContext._active_spark_context is not None:
    raise RuntimeError(
        "Ya existe un SparkContext. Reinicia el kernel "
        "y ejecuta esta celda antes de cualquier otra inicialización."
    )

spark = (
    SedonaContext.builder()
    .appName("american-tower-bronze")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.memoryOverhead", "2g")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.speculation", "false")
    .config("spark.pyspark.python", PYTHON_ENV)
    .config("spark.pyspark.driver.python", PYTHON_ENV)
    .config("spark.executorEnv.PYSPARK_PYTHON", PYTHON_ENV)
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.7.2,"
        "org.datasyslab:geotools-wrapper:1.7.2-28.5"
    )
    .getOrCreate()
)

sc = spark.sparkContext

print("Python del driver:", sys.executable)
print("Python para tareas:", sc.pythonExec)
print("Master:", sc.master)
print("Aplicación:", sc.applicationId)

assert sc.master == "yarn"
assert sc.pythonExec == PYTHON_ENV, sc.pythonExec

# Si falla, conserva el traceback completo.
spark = SedonaContext.create(spark)

spark.sql(
    "SELECT ST_AsText(ST_Point(-77.04, -12.05)) AS geometria"
).show(truncate=False)
import json
def probe_environment(iterator):
    import sys, socket
    import numpy, pyarrow, shapely, geopandas, rasterio, pyproj, pyogrio
    from importlib.metadata import version
    yield {'host':socket.gethostname(), 'python':sys.executable, 'records':sum(1 for _ in iterator),
           'versions':{p:version(p) for p in ['numpy','pyarrow','shapely','geopandas','rasterio','pyproj','pyogrio','apache-sedona']}}
ENV_SAMPLES=sc.parallelize(range(96),12).mapPartitions(probe_environment).collect()
assert sum(r['records'] for r in ENV_SAMPLES)==96
EXPECTED_VERSIONS={'numpy':'1.26.4','pyarrow':'17.0.0','shapely':'2.0.6','geopandas':'1.0.1','rasterio':'1.3.11','pyproj':'3.6.1','pyogrio':'0.10.0','apache-sedona':'1.7.2'}
assert all(r['python']==PYTHON_ENV and r['versions']==EXPECTED_VERSIONS for r in ENV_SAMPLES), ENV_SAMPLES
print(json.dumps(ENV_SAMPLES,indent=2))
print('ENTORNO_DISTRIBUIDO_OK; hosts muestreados:',len({r['host'] for r in ENV_SAMPLES}))
# Este muestreo no obliga al scheduler a utilizar todos los nodos.

print("Spark UI:", sc.uiWebUrl)
print("Ejecutores solicitados: 3; cores por executor: 2; memoria JVM: 4 GiB + overhead 2 GiB")


## 1. Entradas operativas
Complete `OSIPTEL_COBERTURA_INPUT`, `OSIPTEL_ESTACIONES_INPUT` y sus fichas de procedencia; marque `OSIPTEL_PROVENANCE_VERIFIED=True` al contrastarlas. Se aceptan ruta absoluta remota, HTTPS o URI S3. Si el acceso web exige sesión, descargue desde el portal y copie el archivo al primary. No invente fechas: use `None` si no se conocen. La fecha de descarga real sí es obligatoria.

El techo de 8192 GiB es un límite de ejecución, no una estimación del volumen nacional. El preflight obtiene los tamaños y se detiene si lo excede. No amplía automáticamente ese límite ni reduce Perú a una ciudad. Para reanudar use el `PLAN_URI` de una corrida anterior y un kernel nuevo.

In [ ]:
from datetime import datetime

def solicitar_fecha_descarga(nombre):
    valor = input(
        f"Fecha real de descarga de {nombre} "
        "(ejemplo de formato: 2026-10-09T22:30:00-05:00): "
    ).strip()

    fecha = datetime.fromisoformat(valor.replace("Z", "+00:00"))
    if fecha.tzinfo is None:
        raise ValueError("Incluye la zona horaria: -05:00 para Perú o Z para UTC.")

    return fecha.isoformat()

ARTIFACTS_BUCKET='american-tower-dev-artifacts-74d32893263161eaef72faf57a'
CONFIG_KEY='config/emr_ingesta/v1_alineada/project_config.json'
OSIPTEL_COBERTURA_INPUT = (
    "/home/hadoop/dev/datasets/osiptel/archivo_cobertura.csv"
)  # URI S3 de entrada o archivo POSIX visible en el driver remoto
OSIPTEL_ESTACIONES_INPUT = (
    "/home/hadoop/dev/datasets/osiptel/archivo_estaciones.xlsx"
)
OSIPTEL_COBERTURA_NAME=None
OSIPTEL_ESTACIONES_NAME=None
OSIPTEL_OBTAINED_AT=None
PERU_AOI_INPUT=None  # GeoJSON EPSG:4326 derivado de INEI, si no usará extractor RAR
EXTRA_ENTRIES=[]  # Metadatos/archivos originales adicionales, contrato descrito en guía
RESUME_PLAN_URI=None  # s3://Operations/evidence/ingestion/run_id=.../plan.json
CACHE_ROOT='/tmp/american_tower_ingestion'
EXECUTE_INGESTION=True
MAX_FILE_GIB=8
MIN_FREE_GIB=2
MAX_TOTAL_GIB=8192  # techo operativo explícito; no se amplía automáticamente
MAX_INVENTORY_FILES=200000
MAX_STAC_ITEMS_PER_QUARTER=30000
VERIFY_S3_BYTES=True
INEI_DISTRICT_LAYER=None  # si el GPKG tiene varias capas, indique la distrital
OSIPTEL_PROVENANCE_VERIFIED = True  # True después de completar y contrastar las fichas siguientes
OSIPTEL_PROVENANCE = {
    "osiptel_cobertura": {
        "source_url": (
            "https://www.datosabiertos.gob.pe/dataset/"
            "cobertura-de-servicio-movil-por-empresa-operadora"
        ),
        "downloaded_at": solicitar_fecha_descarga("OSIPTEL cobertura"),
        "published_at": None,
        "modified_at": None,
        "observation_period": "202303",
        "period_evidence": (
            "Verificado en el archivo de entrada: la columna PERIODO "
            "contiene únicamente el valor 202303."
        ),
    },
    "osiptel_estaciones": {
        "source_url": (
            "https://www.datosabiertos.gob.pe/dataset/"
            "cantidad-de-estaciones-base-por-tecnolog%C3%ADa-osiptel"
        ),
        "downloaded_at": solicitar_fecha_descarga("OSIPTEL estaciones"),
        "published_at": "2022-09-19",
        "modified_at": "2026-07-24",
        "observation_period": None,
        "period_evidence": (
            "Periodo de observación no determinado. Las fechas de "
            "lanzamiento y modificación fueron reportadas por el usuario "
            "desde el portal y no se interpretan como fechas de observación."
        ),
    },
}

# La ficha documenta expresamente los periodos todavía desconocidos.
# downloaded_at: fecha real ISO 8601 de descarga. source_url: página/recurso oficial.
# Fecha de modificación/lanzamiento NO equivale al trimestre observado.

INGESTION_REVISION='bronze_emr_4'
STAC_COLLECTION='sentinel-2-l2a'
PREFLIGHT_WORKERS=8
STAC_PAGE_SIZE=100
CATALOG_CACHE_ROOT='/tmp/american_tower_ingestion/catalog_cache_v3'
# Cache de catálogo local y durable en Operations, separada de los temporales de archivos.
# No se reanuda un plan v2. Conservará las fechas ingresadas en variables de esta sesión.

# RAR distrital preservado por la corrida adjunta, antes de consultar Sentinel.
# Reutilizarlo mantiene la misma versión territorial y evita una nueva descarga HTTPS.
INEI_DISTRICT_INPUT_URI=(
    's3://american-tower-dev-operations-1dd72256126674a5893dfca3df/'
    'evidence/ingestion/run_id=20261011T012837Z-6495399b/planning_inputs/Distrito.rar'
)

CATALOG_CONCURRENCY=3
SELECTION_PARTITIONS=3
TRANSFER_PARTITIONS=6
TRANSFER_BATCH_SIZE=600
PHASE_METRICS=[]
# Los catálogos siguen usando catálogo cacheado v3. El cache de selección es v4.


## 2. Entorno y contrato compartido
Se rechaza explícitamente la configuración antigua. No depende de memoria de otra notebook.

In [ ]:
from contextlib import closing
import json, os, sys, platform, socket, hashlib, time, uuid, logging, shutil, re
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

def utc(): return datetime.now(timezone.utc).isoformat()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
AWS = boto3.Session()
SDK_CONFIG = Config(retries={'max_attempts': 8, 'mode': 'standard'}, connect_timeout=15, read_timeout=120)
s3 = AWS.client('s3', config=SDK_CONFIG)
def put_json(bucket, key, value):
    return s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(value, ensure_ascii=False, indent=2, default=str).encode(), ContentType='application/json')
def get_json(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    with closing(response['Body']) as body: return json.loads(body.read())
def digest_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
def digest_object(bucket,key,version=None):
    args={'Bucket':bucket,'Key':key}
    if version: args['VersionId']=version
    response=s3.get_object(**args); h=hashlib.sha256(); count=0
    with closing(response['Body']) as body:
        for block in iter(lambda:body.read(8*1024*1024),b''): h.update(block); count+=len(block)
    return h.hexdigest(),count


CONFIG=get_json(ARTIFACTS_BUCKET,CONFIG_KEY)
if CONFIG.get('schema_version')!=2 or CONFIG.get('spec_version')!='1.0.0':
    raise RuntimeError('Ejecute notebook 00 alineada; la configuración antigua no satisface este contrato.')
SPEC=CONFIG['spec']; PERIODS=SPEC['periods']; SENTINEL=SPEC['source_policy']['sentinel']
assert len(PERIODS)==29 and PERIODS[0]=='2019Q1' and PERIODS[-1]=='2026Q1'
identity=AWS.client('sts',config=SDK_CONFIG).get_caller_identity()
if identity['Account']!=CONFIG['account_id']: raise RuntimeError('Cuenta distinta de la verificada en notebook 00')
CACHE=Path(CACHE_ROOT)/RUN_ID; CACHE.mkdir(parents=True,exist_ok=False)
LOG_PATH=CACHE/'ingestion.log'
logger=logging.getLogger('bronze.'+RUN_ID);logger.setLevel(logging.INFO);logger.propagate=False
formatter=logging.Formatter('%(asctime)s %(levelname)s %(message)s')
for handler in [logging.StreamHandler(sys.stdout),logging.FileHandler(LOG_PATH,encoding='utf-8')]:
    handler.setFormatter(formatter);logger.addHandler(handler)
OP_PREFIX=f"{CONFIG['evidence_root']}/ingestion/run_id={RUN_ID}"
INVENTORY=[]; PLANNING_ERRORS=[]; COVERAGE_REPORTS=[]
if not VERIFY_S3_BYTES: raise ValueError('Se exige SHA-256 remoto completo')
print('run_id:',RUN_ID,'config:',CONFIG_KEY,'periodos:',len(PERIODS),'cache remota:',CACHE)
CONFIG['expected_primary_instance']='m5.xlarge'  # corrige la expectativa; no redimensiona EMR
if CONFIG['buckets']['bronze']!='american-tower-dev-bronze-b7f9699786f4c119558b9d30f9':
    raise RuntimeError('Bucket Bronze distinto del auditado.')
ENVIRONMENT={'driver_python':sys.executable,'spark_version':spark.version,'master':sc.master,
             'application_id':sc.applicationId,'executor_samples':ENV_SAMPLES}
if not RESUME_PLAN_URI:
    problems=[]
    for src,locator in [('osiptel_cobertura',OSIPTEL_COBERTURA_INPUT),('osiptel_estaciones',OSIPTEL_ESTACIONES_INPUT)]:
        if not locator: problems.append(src+': falta archivo de entrada')
        p=OSIPTEL_PROVENANCE[src]
        if not p['source_url'] or not p['downloaded_at']: problems.append(src+': falta URL oficial o fecha real de descarga')
        if p['downloaded_at']:
            datetime.fromisoformat(p['downloaded_at'].replace('Z','+00:00'))
        if p['observation_period'] and not p['period_evidence']: problems.append(src+': falta evidencia del periodo observado')
    if not OSIPTEL_PROVENANCE_VERIFIED: problems.append('Marque OSIPTEL_PROVENANCE_VERIFIED después de contrastar las fichas.')
    if problems: raise ValueError('\n'.join(problems))
print('CONTRATO_Y_ENTRADAS_OK')

for src,locator in [('osiptel_cobertura',OSIPTEL_COBERTURA_INPUT),('osiptel_estaciones',OSIPTEL_ESTACIONES_INPUT)]:
    if not RESUME_PLAN_URI and not urlparse(locator).scheme and not Path(locator).is_file():
        raise FileNotFoundError(f'{src}: la ruta no existe en el primary: {locator}')
CONFIG['ingestion_implementation']={'revision':INGESTION_REVISION,'stac_collection':STAC_COLLECTION,
 'selection':'maximum_national_footprint_then_cloud_with_monthly_replacements',
 'maximum_scenes_per_tile_quarter':3,'max_total_gib':MAX_TOTAL_GIB,
 'metadata_storage':'one_full_response_jsonl_gzip_per_quarter',
 'changes_from_v2':['collection','coverage_first_selection','metadata_batching','early_validation','parallel_preflight']}

CONFIG['ingestion_implementation']['asset_resolution_revision']='3.1'
CONFIG['ingestion_implementation']['asset_access_policy']='public_https_bands_and_documented_metadata_proxy'

CONFIG['ingestion_implementation'].update({'execution_revision':'4.0','selection_execution':'spark_mapPartitions_by_quarter','transfer_execution':'spark_mapPartitions_one_payload_per_host','catalog_concurrency':CATALOG_CONCURRENCY,'selection_partitions':SELECTION_PARTITIONS,'transfer_partitions':TRANSFER_PARTITIONS})

if not (1<=CATALOG_CONCURRENCY<=3 and 1<=SELECTION_PARTITIONS<=6 and 1<=TRANSFER_PARTITIONS<=12 and 1<=TRANSFER_BATCH_SIZE<=1200):raise ValueError('Concurrencia fuera del límite documentado; revisar parámetros antes de continuar.')


### 2.1. Permisos reales sobre Bronze
La prueba deja un objeto diagnóstico de 9 MiB en `_preflight`. Comprueba multipart y lectura íntegra SHA-256; `HeadBucket` por sí solo no prueba escritura. Operations recibe la evidencia. Si falla, conserve el traceback antes de continuar.

In [ ]:
import io
from boto3.s3.transfer import TransferConfig

for zone in ['artifacts','operations','bronze']:
    s3.head_bucket(Bucket=CONFIG['buckets'][zone])
payload=b'\x00'*(9*1024*1024)
probe_key=f"{CONFIG['bronze_root']}/_preflight/run_id={RUN_ID}/multipart_probe.bin"
s3.upload_fileobj(io.BytesIO(payload),CONFIG['buckets']['bronze'],probe_key,
    ExtraArgs={'ContentType':'application/octet-stream'},
    Config=TransferConfig(multipart_threshold=5*1024*1024,multipart_chunksize=5*1024*1024,max_concurrency=2))
response = s3.get_object(Bucket=CONFIG['buckets']['bronze'], Key=probe_key)
body = response['Body']

sha = hashlib.sha256()
count = 0

if hasattr(body, "iter_chunks"):
    for chunk in body.iter_chunks(chunk_size=8 * 1024 * 1024):
        sha.update(chunk)
        count += len(chunk)
else:
    while True:
        chunk = body.read(8 * 1024 * 1024)
        if not chunk:
            break
        sha.update(chunk)
        count += len(chunk)

probe_sha = sha.hexdigest()
probe_bytes = count
assert probe_bytes==len(payload) and probe_sha==hashlib.sha256(payload).hexdigest()
BRONZE_PREFLIGHT={'ok':True,'bytes':probe_bytes,'sha256':probe_sha,'uri':f"s3://{CONFIG['buckets']['bronze']}/{probe_key}"}
put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/environment_preflight.json',
         {'environment':ENVIRONMENT,'bronze':BRONZE_PREFLIGHT,'at':utc()})
print('BRONZE_MULTIPART_Y_SHA256_OK',BRONZE_PREFLIGHT['uri'])


## 3. Clientes, selección mensual y reanudación
Metadatos de proveedor se preservan como JSON completo; los derivados se identifican por zona. El plan congelado referencia S3, no temporales de otra sesión.

In [ ]:
from botocore import UNSIGNED
from boto3.s3.transfer import TransferConfig
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
PUBLIC_S3=boto3.client('s3',region_name='us-east-1',config=Config(signature_version=UNSIGNED,retries={'max_attempts':8,'mode':'standard'}))
TRANSFER=TransferConfig(multipart_threshold=64*1024*1024,multipart_chunksize=64*1024*1024,max_concurrency=3,use_threads=True)
HTTP=requests.Session()
HTTP.headers.update({'User-Agent':'AmericanTower-Academic-Bronze/1.0','Accept-Encoding':'identity'})
HTTP.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504],allowed_methods=['GET','HEAD'])))

def entry(source,locator,name,kind='data',period=None,publisher=None,license=None,provenance=None,**extra):
    return {'source':source,'locator':locator,'original_name':name,'kind':kind,'observation_period':period,'obtained_at':None,'publisher':publisher,'license':license,'provenance_url':provenance,**extra}
def local_json(name,value):
    path=CACHE/name; path.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str),encoding='utf-8'); return str(path)
def json_url(url):
    if urlparse(url).scheme!='https': raise ValueError('Se exige HTTPS')
    with HTTP.get(url,timeout=(15,120)) as response:
        response.raise_for_status()
        if urlparse(response.url).scheme!='https': raise ValueError('Redirección insegura')
        return response.json()
def s3_parts(uri):
    p=urlparse(uri)
    if p.scheme!='s3' or not p.netloc or not p.path.strip('/'): raise ValueError('URI S3 inválida')
    return p.netloc,p.path.lstrip('/')
def planning_error(source,period,exc):
    error={'source':source,'period':period,'error_type':type(exc).__name__,'at':utc(),'reason':str(exc)[:400]}
    PLANNING_ERRORS.append(error);logger.error('Planificación %s %s: %s · %s',source,period,error['error_type'],error['reason'])
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/planning_errors.json',PLANNING_ERRORS)
def save_planning_metadata(source,name,value,period=None,derived=False,**extra):
    # Persistir antes de congelar el plan: la reanudación no depende de /tmp de otra sesión.
    key=OP_PREFIX+('/derived/' if derived else '/provider_metadata/')+source+'/'+name
    serialized=json.dumps(value,ensure_ascii=False,indent=2,default=str).encode()
    uploaded=s3.put_object(Bucket=CONFIG['buckets']['operations'],Key=key,Body=serialized,ContentType='application/json')
    return entry(source,f"s3://{CONFIG['buckets']['operations']}/{key}",name,'metadata',period=period,
                 publisher='Pipeline del proyecto' if derived else extra.pop('publisher','Proveedor de la fuente'),
                 license=extra.pop('license','conservar_atribucion'),provenance=extra.pop('provenance',f'https://earth-search.aws.element84.com/v1/collections/{STAC_COLLECTION}' if source=='sentinel' else 'urn:american-tower:documented-provenance'),
                 target_zone='operations' if derived else 'bronze',derived=derived,expected_bytes=len(serialized),expected_sha256=hashlib.sha256(serialized).hexdigest(),input_version_id=uploaded.get('VersionId'),staged_in_current_run=True,**extra)
import gzip
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlencode, urljoin

def archive_entry(source,name,path,period=None):
    size=Path(path).stat().st_size; sha=digest_file(path)
    key=f'{OP_PREFIX}/provider_metadata/{source}/{name}'
    s3.upload_file(str(path),CONFIG['buckets']['operations'],key,Config=TRANSFER)
    head=s3.head_object(Bucket=CONFIG['buckets']['operations'],Key=key)
    return entry(source,f"s3://{CONFIG['buckets']['operations']}/{key}",name,'metadata',period=period,
        publisher='Element84 STAC; Copernicus/ESA',license='Copernicus_Sentinel_legal_notice',
        provenance=f'https://earth-search.aws.element84.com/v1/collections/{STAC_COLLECTION}',
        expected_bytes=size,planned_bytes=size,expected_sha256=sha,input_version_id=head.get('VersionId'),
        staged_in_current_run=True,metadata_origin='complete_STAC_responses_archived_JSONL_gzip')

def discover_pages(query,period,http=None):
    http=http or HTTP
    url='https://earth-search.aws.element84.com/v1/search?'+urlencode(query)
    method='GET';body=None;visited=set();index=0
    while url:
        token=(method,url,json.dumps(body,sort_keys=True))
        if token in visited:raise RuntimeError('Paginación STAC cíclica')
        visited.add(token)
        response=None
        for attempt in range(4):
            try:
                response=http.request(method,url,json=body if method=='POST' else None,timeout=(15,120))
                response.raise_for_status();page=response.json();break
            except (requests.RequestException,ValueError):
                if response is not None:response.close()
                if attempt==3:raise
                time.sleep(2*(attempt+1))
        if urlparse(response.url).scheme!='https':response.close();raise ValueError('STAC no HTTPS')
        response.close()
        yield index,page,url
        link=next((l for l in page.get('links',[]) if l.get('rel')=='next'),None)
        if not link:break
        url=urljoin(url,link['href']);method=link.get('method','GET').upper();body=link.get('body')
        if link.get('merge'):body={**query,**(body or {})}
        if urlparse(url).scheme!='https' or method not in ['GET','POST']:raise ValueError('next STAC inválido')
        index+=1


def public_sentinel_bands(item,bands):
    """Only public primary band assets enter selection; complete Items remain archived."""
    return all(urlparse(item.get('assets',{}).get(b,{}).get('href','')).scheme=='https' for b in bands)

_SENTINEL_PRODUCT_INFO={}
def resolve_sentinel_asset(item,asset_key,asset,is_band):
    """Resolve documented metadata proxy; never enable Requester Pays or invent COG paths."""
    href=asset.get('href','');parsed=urlparse(href)
    if parsed.scheme=='https':return href,{'method':'catalog_https','catalog_href':href}
    label=f"scene={item.get('id')} asset={asset_key} href={href}"
    if is_band:raise RuntimeError('Banda sin HTTPS público: '+label)
    if parsed.scheme!='s3' or parsed.netloc!='sentinel-s2-l2a':
        raise RuntimeError('Origen de metadato no soportado: '+label)
    key=parsed.path.lstrip('/')
    if not key.lower().endswith(('.xml','.json')) or '..' in key.split('/'):
        raise RuntimeError('Ruta de metadato inválida: '+label)
    base='https://roda.sentinel-hub.com/sentinel-s2-l2a/'
    evidence={'catalog_href':href,'method':'sentinel_hub_metadata_proxy','requester_pays':False}
    if asset_key=='product_metadata' and re.fullmatch(r'tiles/[^/]+/[^/]+/[^/]+/\d{4}/\d{1,2}/\d{1,2}/\d+/product_metadata\.xml',key):
        info_url=base+key.rsplit('/',1)[0]+'/productInfo.json'
        if info_url not in _SENTINEL_PRODUCT_INFO:_SENTINEL_PRODUCT_INFO[info_url]=json_url(info_url)
        info=_SENTINEL_PRODUCT_INFO[info_url];product_path=info.get('path','')
        if not re.fullmatch(r'products/\d{4}/\d{1,2}/\d{1,2}/[A-Za-z0-9_.-]+',product_path):
            raise RuntimeError('productInfo.path inválido: '+label)
        expected=item.get('properties',{}).get('s2:product_uri','').removesuffix('.SAFE')
        if expected and product_path.rsplit('/',1)[-1]!=expected:
            raise RuntimeError('productInfo no corresponde al producto STAC: '+label)
        key=product_path+'/metadata.xml'
        evidence.update({'method':'sentinel_hub_productInfo_path','product_info_url':info_url,'product_info':info})
    elif not key.startswith(('tiles/','products/')):
        raise RuntimeError('Ruta de metadato fuera del dataset: '+label)
    evidence['resolved_href']=base+key
    return base+key,evidence

SELECTION_SOURCE = r"""
from collections import defaultdict
from datetime import datetime
import re
from shapely.geometry import shape
from shapely.ops import unary_union, transform
from shapely.prepared import prep
from shapely import set_precision
from shapely.strtree import STRtree
from pyproj import Transformer

def mgrs(item):
    p=item['properties']
    if p.get('s2:mgrs_tile'):return str(p['s2:mgrs_tile'])
    if p.get('grid:code','').startswith('MGRS-'):return p['grid:code'][5:]
    if all(k in p for k in ['mgrs:utm_zone','mgrs:latitude_band','mgrs:grid_square']):
        return f"{int(p['mgrs:utm_zone']):02d}{p['mgrs:latitude_band']}{p['mgrs:grid_square']}"
    m=re.search(r'_T?(\d{2}[A-Z]{3})_',item['id'])
    if not m:raise ValueError('Item STAC sin tesela MGRS reconocible')
    return m.group(1)

def select_coverage_first(items,period,national,bands,required_fraction):
    project=Transformer.from_crs(4326,6933,always_xy=True).transform
    region=set_precision(transform(project,national),0.01); area=region.area
    if area<=0:raise ValueError('AOI sin área')
    groups=defaultdict(list); footprints={}; candidates={}
    quarter=int(period[-1]);year=int(period[:4]);months=range(3*(quarter-1)+1,3*quarter+1)
    # Exact footprint area on national AOI. Bounding-box quick rejection precedes intersection.
    for item in items:
        if not all(b in item.get('assets',{}) for b in bands):continue
        date=datetime.fromisoformat(item['properties']['datetime'].replace('Z','+00:00'))
        if date.year!=year or date.month not in months:raise ValueError('Escena fuera del trimestre')
        geom=shape(item['geometry'])
        if geom.is_empty or not geom.is_valid:raise ValueError('Huella STAC inválida')
        clip=set_precision(transform(project,geom),0.01).intersection(region)
        if clip.area<=0:continue
        candidates[item['id']]=item;footprints[item['id']]=clip
        groups[(mgrs(item),date.month)].append(item)
    if not groups:raise RuntimeError('Sin candidatos elegibles con bandas completas')
    def cloud_key(item):
        c=item['properties'].get('eo:cloud_cover')
        return (float(c) if c is not None else 101,item['properties']['datetime'],item['id'])
    selected={}
    for key,group in sorted(groups.items()):
        max_area=max(footprints[item['id']].area for item in group)
        # Near ties are limited to 1e-8 relative area; clouds never outrank a partial swath.
        eligible=[i for i in group if footprints[i['id']].area>=max_area*(1-1e-8)]
        selected[key]=min(eligible,key=cloud_key)
    def coverage(current):return unary_union([footprints[i['id']] for i in current.values()])
    by_tile=defaultdict(list)
    for item in candidates.values():by_tile[mgrs(item)].append(footprints[item['id']])
    all_footprint=unary_union([unary_union(group) for group in by_tile.values()])
    candidate_fraction=min(1.0,all_footprint.area/area)
    merged=coverage(selected)
    print(period,'cobertura inicial %',round(100*merged.area/area,6),'catálogo %',round(100*candidate_fraction,6),flush=True)
    # Replacement only: retain at most one acquisition per tile-month and three per quarter.
    swaps=0
    while candidate_fraction>=required_fraction and merged.area/area<required_fraction and swaps<max(100,2*len(groups)):
        uncovered=region.difference(merged); proposals=[];prepared_uncovered=prep(uncovered)
        for key,group in groups.items():
            current=selected[key]
            for item in group:
                if item['id']==current['id']:continue
                if not prepared_uncovered.intersects(footprints[item['id']]):continue
                gain=footprints[item['id']].intersection(uncovered).area
                if gain>area*1e-10:proposals.append((gain,key,item))
        selected_keys=list(selected);selected_shapes=[footprints[selected[k]['id']] for k in selected_keys]
        tree=STRtree(selected_shapes);exclusive_by_key={};best=None;best_area=merged.area
        # Only neighboring selected footprints can overlap the scene being replaced.
        # This computes exactly the lost area without repeatedly unioning all of Peru.
        for gain,key,item in sorted(proposals,key=lambda x:-x[0]):
            if key not in exclusive_by_key:
                old=footprints[selected[key]['id']]
                neighbors=[selected_shapes[int(j)] for j in tree.query(old) if selected_keys[int(j)]!=key]
                exclusive_by_key[key]=old.difference(unary_union(neighbors)) if neighbors else old
            loss=exclusive_by_key[key].difference(footprints[item['id']]).area
            new_area=merged.area+gain-loss
            if new_area>best_area+area*1e-10:best=(key,item);best_area=new_area
        if best is None:break
        key,item=best;selected[key]=item;merged=coverage(selected);swaps+=1
        if swaps%10==0:print(period,'reemplazos',swaps,'cobertura %',round(100*merged.area/area,6),flush=True)
    fraction=min(1.0,merged.area/area)
    return selected,{'candidate_footprint_coverage':candidate_fraction,
                     'national_footprint_coverage':fraction,'repair_swaps':swaps,'coverage_precision_m':0.01,
                     'eligible_candidate_count':len(candidates),
                     'missing_fraction':max(0,1-fraction)},region.difference(merged),project
"""
CONFIG["ingestion_implementation"]["selection_code_sha256"]=hashlib.sha256(SELECTION_SOURCE.encode()).hexdigest()

if RESUME_PLAN_URI:
    b,k=s3_parts(RESUME_PLAN_URI);saved=get_json(b,k)
    if saved.get('ingestion_revision')!=INGESTION_REVISION: raise ValueError('Plan anterior incompatible: requiere planificación distribuida v4; catálogos v3 reutilizables.')
    if saved.get('spec_version')!='1.0.0' or saved.get('schema_version')!=2: raise ValueError('Plan antiguo o incompatible')
    if saved['config']['account_id']!=identity['Account']: raise ValueError('Cuenta del plan distinta')
    # La reanudación usa exactamente el contrato/configuración y entradas del plan congelado.
    if saved.get('config',{}).get('ingestion_implementation',{}).get('asset_resolution_revision')!='3.1':raise ValueError('Plan sin corrección de assets 3.1; regenerar planificación usando catálogo cacheado.')
    if saved['planning_errors']: raise ValueError('No se reanuda un plan con errores de planificación.')
    CONFIG=saved['config'];SPEC=CONFIG['spec'];SENTINEL=SPEC['source_policy']['sentinel'];PERIODS=SPEC['periods']
    INVENTORY=saved['entries'];PLANNING_ERRORS=saved['planning_errors'];COVERAGE_REPORTS=saved['coverage_reports']
    print('Reanudando inventario congelado; no se vuelve a consultar STAC/Ookla.')
# Embedded in notebook: only broadcasts of plain configuration/code and S3 locators.
def collect_with_progress(sc,rdd,label,interval=15):
    from concurrent.futures import ThreadPoolExecutor,TimeoutError
    import time
    group=label+'-'+str(time.time_ns());pool=ThreadPoolExecutor(max_workers=1)
    def run():
        sc.setJobGroup(group,label,interruptOnCancel=True)
        try:return rdd.collect()
        finally:sc.setLocalProperty('spark.jobGroup.id',None)
    future=pool.submit(run);started=time.perf_counter()
    try:
        while True:
            try:return future.result(timeout=interval)
            except TimeoutError:
                stages=[];tracker=sc.statusTracker()
                for job_id in tracker.getJobIdsForGroup(group):
                    job=tracker.getJobInfo(job_id)
                    if job:
                        for stage_id in job.stageIds:
                            stage=tracker.getStageInfo(stage_id)
                            if stage:stages.append({'stage':stage_id,'total':stage.numTasks,'activas':stage.numActiveTasks,'completas':stage.numCompletedTasks})
                print(label,'segundos',round(time.perf_counter()-started,1),'Spark',stages,flush=True)
    except BaseException:
        sc.cancelJobGroup(group);raise
    finally:pool.shutdown(wait=False)


def selection_worker_factory(settings_broadcast):
    def process(iterator):
        import boto3,json,gzip,tempfile,hashlib,time,socket,sys,os
        from pathlib import Path
        from urllib.parse import urlparse
        from botocore.config import Config
        from botocore.exceptions import ClientError
        from shapely import wkb
        from shapely.geometry import shape,mapping
        from shapely.ops import transform
        from pyproj import Transformer
        cfg=settings_broadcast.value
        client=boto3.session.Session().client('s3',config=Config(retries={'max_attempts':8,'mode':'standard'},connect_timeout=15,read_timeout=120))
        namespace={};exec(cfg['selector_source'],namespace)
        national=wkb.loads(bytes.fromhex(cfg['aoi_wkb']))
        def sha(path):
            h=hashlib.sha256()
            with open(path,'rb') as f:
                for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
            return h.hexdigest()
        for job in iterator:
            start=time.perf_counter();period=job['period']
            try:
                root=f"selection_cache/bronze_emr_4/{job['selection_id']}"
                report_key=root+'/report.json';bucket=cfg['operations_bucket']
                cached=None
                try:
                    obj=client.get_object(Bucket=bucket,Key=report_key)
                    with obj['Body'] as body:cached=json.loads(body.read())
                except ClientError as exc:
                    if exc.response['Error']['Code'] not in ['404','NoSuchKey','NotFound']:raise
                with tempfile.TemporaryDirectory(prefix='at-selection-') as temp:
                    temp=Path(temp);selected_path=temp/'selected.jsonl.gz'
                    if cached and cached.get('complete') and cached.get('selection_id')==job['selection_id']:
                        client.download_file(bucket,cached['selected_key'],str(selected_path))
                        if sha(selected_path)!=cached['selected_sha256']:raise ValueError('Cache de selección corrupto')
                        yield {**cached,'status':'ok','cache_hit':True,'worker_host':socket.gethostname(),'worker_python':sys.executable,'worker_pid':os.getpid(),'seconds':round(time.perf_counter()-start,3)}
                        continue
                    catalog=temp/'catalog.jsonl.gz';client.download_file(bucket,job['catalog_key'],str(catalog))
                    if sha(catalog)!=job['catalog_sha256']:raise ValueError('Catálogo remoto corrupto')
                    read_done=time.perf_counter();items={};excluded=set()
                    with gzip.open(catalog,'rt',encoding='utf-8') as pages:
                        for line in pages:
                            for item in json.loads(line).get('features',[]):
                                cloud=item['properties'].get('eo:cloud_cover')
                                if cloud is not None and float(cloud)>cfg['max_scene_cloud_pct']:continue
                                if not shape(item['geometry']).intersects(national):continue
                                if not all(urlparse(item.get('assets',{}).get(b,{}).get('href','')).scheme=='https' for b in cfg['bands']):excluded.add(item['id']);continue
                                items[item['id']]={'id':item['id'],'geometry':item['geometry'],'properties':item['properties'],'assets':{b:{} for b in cfg['bands']}}
                    filter_done=time.perf_counter()
                    selected,coverage,gap,_=namespace['select_coverage_first'](items.values(),period,national,cfg['bands'],cfg['required_fraction'])
                    geometry_done=time.perf_counter();wanted={i['id'] for i in selected.values()};found=set()
                    with gzip.open(selected_path,'wt',encoding='utf-8') as out,gzip.open(catalog,'rt',encoding='utf-8') as pages:
                        for line in pages:
                            for item in json.loads(line).get('features',[]):
                                if item['id'] in wanted and item['id'] not in found:out.write(json.dumps(item,ensure_ascii=False)+'\n');found.add(item['id'])
                    if found!=wanted:raise ValueError('Items seleccionados ausentes en catálogo')
                    inverse=Transformer.from_crs(6933,4326,always_xy=True).transform
                    gap_value={'type':'Feature','properties':{'period':period,'derived':True},'geometry':mapping(transform(inverse,gap))}
                    selected_key=root+'/selected_items.jsonl.gz';client.upload_file(str(selected_path),bucket,selected_key)
                    report={'complete':True,'status':'ok','selection_id':job['selection_id'],'period':period,
                            'candidate_count':len(items),'selected_count':len(found),'selected_ids':sorted(found),**coverage,
                            'non_public_band_item_ids':sorted(excluded),'non_public_band_item_count':len(excluded),
                            'selected_key':selected_key,'selected_sha256':sha(selected_path),'uncovered':gap_value,
                            'months_selected':sorted({(namespace['mgrs'](item),int(item['properties']['datetime'][5:7])) for item in selected.values()}),
                            'phase_seconds':{'catalog_read':read_done-start,'filter':filter_done-read_done,'geometry_selection':geometry_done-filter_done,'persist':time.perf_counter()-geometry_done},
                            'worker_host':socket.gethostname(),'worker_python':sys.executable,'worker_pid':os.getpid(),'seconds':round(time.perf_counter()-start,3),'cache_hit':False}
                    client.put_object(Bucket=bucket,Key=report_key,Body=json.dumps(report,ensure_ascii=False).encode(),ContentType='application/json')
                    yield report
            except Exception as exc:
                yield {'period':period,'status':'failed','error_type':type(exc).__name__,'reason':str(exc)[:600],'worker_host':socket.gethostname(),'seconds':round(time.perf_counter()-start,3)}
    return process


def transfer_worker_factory(settings_broadcast):
    def process(iterator):
        import boto3,json,tempfile,hashlib,time,socket,sys,os,logging,shutil,gzip,fcntl,requests
        from pathlib import Path
        from contextlib import closing
        from datetime import datetime,timezone
        from urllib.parse import urlparse
        from botocore import UNSIGNED
        from botocore.config import Config
        from botocore.exceptions import ClientError
        from boto3.s3.transfer import TransferConfig
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry
        cfg=settings_broadcast.value;session=boto3.session.Session()
        sdk=Config(retries={'max_attempts':8,'mode':'standard'},connect_timeout=15,read_timeout=120)
        client=session.client('s3',config=sdk)
        public=session.client('s3',region_name='us-east-1',config=Config(signature_version=UNSIGNED,retries={'max_attempts':8,'mode':'standard'}))
        http=requests.Session();http.headers.update({'Accept-Encoding':'identity','User-Agent':'AmericanTower-Academic-Bronze/4.0'})
        http.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504],allowed_methods=['GET','HEAD'])))
        log=logging.getLogger('bronze-worker');log.setLevel(logging.INFO)
        def utc():return datetime.now(timezone.utc).isoformat()
        def s3_parts(uri):
            p=urlparse(uri)
            if p.scheme!='s3' or not p.netloc or not p.path.strip('/'):raise ValueError('URI S3 inválida')
            return p.netloc,p.path.lstrip('/')
        def digest_file(path):
            h=hashlib.sha256()
            with open(path,'rb') as f:
                for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
            return h.hexdigest()
        def digest_object(bucket,key,version=None):
            args={'Bucket':bucket,'Key':key}
            if version:args['VersionId']=version
            obj=client.get_object(**args);h=hashlib.sha256();count=0
            with closing(obj['Body']) as body:
                for block in iter(lambda:body.read(8*1024**2),b''):h.update(block);count+=len(block)
            return h.hexdigest(),count
        with tempfile.TemporaryDirectory(prefix='at-transfer-') as temp:
            namespace={'requests':requests,'ClientError':ClientError,'Path':Path,'urlparse':urlparse,'time':time,'json':json,'gzip':gzip,
                       'closing':closing,'shutil':shutil,'logger':log,'CACHE':Path(temp),'MAX_FILE_GIB':cfg['max_file_gib'],'MIN_FREE_GIB':cfg['min_free_gib'],
                       's3':client,'PUBLIC_S3':public,'HTTP':http,'TRANSFER':TransferConfig(multipart_threshold=64*1024**2,multipart_chunksize=64*1024**2,max_concurrency=2),
                       'RUN_ID':cfg['run_id'],'CONFIG':cfg['config'],'OP_PREFIX':cfg['op_prefix'],'utc':utc,'s3_parts':s3_parts,'digest_file':digest_file,'digest_object':digest_object}
            exec(cfg['transfer_source'],namespace)
            # One large temporary payload per host, even if multiple tasks land on the same node.
            with open('/tmp/american_tower_bronze_transfer.lock','a') as lock:
                for index,row in iterator:
                    if urlparse(row['locator']).scheme not in ['https','s3']:raise ValueError('Plan no portable: locator local en executor')
                    fcntl.flock(lock.fileno(),fcntl.LOCK_EX)
                    try:result=namespace['ingest_one'](row,index)
                    finally:fcntl.flock(lock.fileno(),fcntl.LOCK_UN)
                    result.update(worker_host=socket.gethostname(),worker_python=sys.executable,worker_pid=os.getpid())
                    receipt=f"{cfg['op_prefix']}/receipts/{index:05d}.json"
                    client.put_object(Bucket=cfg['operations_bucket'],Key=receipt,Body=json.dumps(result,ensure_ascii=False,default=str).encode(),ContentType='application/json')
                    yield result
        http.close()
    return process


def worker_access_probe_factory(cfg):
    def probe(iterator):
        import boto3,socket,hashlib,shutil,sys,uuid
        list(iterator);client=boto3.session.Session().client('s3');payload=b'american-tower-worker-probe'*2048;digest=hashlib.sha256(payload).hexdigest()
        for zone in ['operations','bronze']:
            key=cfg['op_prefix']+'/worker_preflight/'+zone+'/'+uuid.uuid4().hex+'.bin'
            client.put_object(Bucket=cfg['buckets'][zone],Key=key,Body=payload)
            obj=client.get_object(Bucket=cfg['buckets'][zone],Key=key)
            with obj['Body'] as body:assert hashlib.sha256(body.read()).hexdigest()==digest
        free=shutil.disk_usage('/tmp').free
        if free<(cfg['max_file_gib']+cfg['min_free_gib'])*1024**3:raise RuntimeError('Disco local insuficiente en worker')
        yield {'host':socket.gethostname(),'python':sys.executable,'free_bytes':free,'s3_write_read_sha256':True}
    return probe


## 4. Fuentes nacionales tabulares y WorldPop
Registra originales y documentación del proveedor. Las entradas OSIPTEL se comprobaron al inicio. No se declara una ingesta nacional completa si falta una fuente.

In [ ]:
if not RESUME_PLAN_URI:

    for source,locator,name,period,prov in [
        ('osiptel_cobertura',OSIPTEL_COBERTURA_INPUT,OSIPTEL_COBERTURA_NAME,'202303','https://www.datosabiertos.gob.pe/dataset/cobertura-de-servicio-movil-por-empresa-operadora'),
        ('osiptel_estaciones',OSIPTEL_ESTACIONES_INPUT,OSIPTEL_ESTACIONES_NAME,None,'https://www.datosabiertos.gob.pe/dataset/cantidad-de-estaciones-base-por-tecnolog%C3%ADa-osiptel')]:
        if not locator:
            planning_error(source,None,ValueError('Falta entrada manual OSIPTEL'));continue
        original=name or Path(urlparse(locator).path).name
        INVENTORY.append(entry(source,locator,original,period=OSIPTEL_PROVENANCE[source]['observation_period'],publisher='OSIPTEL',license='licencia_portal_pendiente_verificacion',provenance=OSIPTEL_PROVENANCE[source]['source_url'],obtained_at=OSIPTEL_PROVENANCE[source]['downloaded_at'],delivery_method='manual',original_period_verified=True,period_basis=OSIPTEL_PROVENANCE[source]['period_evidence'],provider_dates=OSIPTEL_PROVENANCE[source]))
    for level in ['Departamento','Provincia','Distrito']:
        url=f'https://ide.inei.gob.pe/files/{level}.rar'
        INVENTORY.append(entry('inei',url,level+'.rar',period='version_anunciada_2023',publisher='INEI',license='condiciones_especificas_pendientes_verificacion',provenance='https://ide.inei.gob.pe',level=level.lower()))
    INVENTORY.append(entry('worldpop','https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/PER/per_ppp_2020.tif','per_ppp_2020.tif',period='2020',publisher='WorldPop',license='CC-BY-4.0',provenance='https://hub.worldpop.org/geodata/summary?id=6416'))
    url='https://hub.worldpop.org/tabs/gdata/html/6416/report_prj_2020_PER.html'
    INVENTORY.append(entry('worldpop',url,'report_prj_2020_PER.html','metadata',publisher='WorldPop',license='CC-BY-4.0',provenance=url))
print('Entradas hasta ahora:',len(INVENTORY))
if not RESUME_PLAN_URI:
    for source,url,name,publisher,license in [
        ('inei','https://ide.inei.gob.pe','geoportal_inei.html','INEI','condiciones_especificas_pendientes_verificacion'),
        ('worldpop','https://hub.worldpop.org/geodata/summary?id=6416','ficha_worldpop.html','WorldPop','CC-BY-4.0'),
        ('osiptel_cobertura','https://www.datosabiertos.gob.pe/dataset/cobertura-de-servicio-movil-por-empresa-operadora','ficha_cobertura.html','OSIPTEL','ODC-By'),
        ('osiptel_estaciones','https://www.datosabiertos.gob.pe/dataset/cantidad-de-estaciones-base-por-tecnolog%C3%ADa-osiptel','ficha_estaciones.html','OSIPTEL','ODC-By')]:
        INVENTORY.append(entry(source,url,name,'metadata',publisher=publisher,license=license,provenance=url))
if not RESUME_PLAN_URI:
    for src in ['osiptel_cobertura','osiptel_estaciones']:
        INVENTORY.append(save_planning_metadata(src,'provenance_declared.json',OSIPTEL_PROVENANCE[src],derived=True,metadata_origin='ficha_documentada_por_equipo'))


## 5. Ookla, todos los trimestres contratados
Ausencia de un trimestre queda documentada y bloquea el éxito global. Los listados se paginan.

In [ ]:
if not RESUME_PLAN_URI:
    for period in PERIODS:
        try:
            year=int(period[:4]);quarter=int(period[-1])
            prefix=f'parquet/performance/type=mobile/year={year}/quarter={quarter}/'
            pages=list(PUBLIC_S3.get_paginator('list_objects_v2').paginate(Bucket='ookla-open-data',Prefix=prefix))
            objects=[o for page in pages for o in page.get('Contents',[]) if o['Key'].endswith('.parquet')]
            if not objects: raise RuntimeError('Trimestre Ookla sin archivos')
            for obj in objects:
                INVENTORY.append(entry('ookla',f"s3://ookla-open-data/{obj['Key']}",Path(obj['Key']).name,period=period,publisher='Ookla',license='CC-BY-NC-SA-4.0',provenance='https://github.com/teamookla/ookla-open-data',anonymous=True,expected_bytes=obj['Size'],source_etag=obj['ETag'],source_key=obj['Key']))
            INVENTORY.append(save_planning_metadata('ookla','listing_'+period+'.json',pages,period,publisher='AWS S3 / Ookla',license='CC-BY-NC-SA-4.0',provenance=f's3://ookla-open-data/{prefix}'))
        except Exception as exc: planning_error('ookla',period,exc)
    INVENTORY.append(entry('ookla','https://raw.githubusercontent.com/teamookla/ookla-open-data/master/README.md','README_proveedor.md','metadata',publisher='Ookla',license='CC-BY-NC-SA-4.0',provenance='https://github.com/teamookla/ookla-open-data'))
print('Inventario Ookla acumulado:',len(INVENTORY))

## 6. Catálogo concurrente y selección distribuida Spark
Reutiliza el cache v3, consulta hasta tres trimestres simultáneos y distribuye un trimestre por tarea Spark. Las uniones/reemplazos Shapely corren en los executors; Sedona permanece disponible para SQL. Cada 15 segundos se informa el estado de tareas. El progreso detallado de reemplazos está en logs de executors. Se guarda selección íntegra verificada por hash para no repetir geometría al reintentar. No baja el umbral ni cambia los periodos.


In [ ]:

def prepare_catalog(period):
    start=time.perf_counter()
    http=requests.Session();http.headers.update({'Accept-Encoding':'identity','User-Agent':'AmericanTower-Academic-Bronze/4.0'})
    http.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504],allowed_methods=['GET','HEAD'])))
    try:
        year=int(period[:4]);quarter=int(period[-1]);month=3*(quarter-1)+1;end_month=month+2
        interval=f'{year}-{month:02d}-01T00:00:00Z/{year}-{end_month:02d}-{calendar.monthrange(year,end_month)[1]}T23:59:59Z'
        query={'collections':STAC_COLLECTION,'bbox':','.join(map(str,national.bounds)),
               'datetime':interval,'limit':STAC_PAGE_SIZE}
        # Cloud filtering is explicit locally; null cloud values remain eligible and rank last.
        query_id=hashlib.sha256(json.dumps(query,sort_keys=True).encode()).hexdigest()
        provider_path=catalog_cache/f'{period}_{query_id}.jsonl.gz'
        cache_meta=provider_path.with_suffix('.cache.json')
        cache_key=f"catalog_cache/bronze_emr_3/{STAC_COLLECTION}/{query_id}.jsonl.gz"
        cache_meta_key=cache_key+'.json'
        cache_valid=False
        if cache_meta.exists() and provider_path.exists():
            cached=json.loads(cache_meta.read_text())
            cache_valid=cached.get('complete') and cached.get('sha256')==digest_file(provider_path)
        if not cache_valid:
            try:
                cached=get_json(CONFIG['buckets']['operations'],cache_meta_key)
                if cached.get('complete'):
                    s3.download_file(CONFIG['buckets']['operations'],cache_key,str(provider_path))
                    cache_valid=digest_file(provider_path)==cached['sha256']
                    if cache_valid:cache_meta.write_text(json.dumps(cached))
            except ClientError as exc:
                if exc.response['Error']['Code'] not in ['404','NoSuchKey','NotFound']:raise
        if not cache_valid:
            tmp=provider_path.with_suffix('.partial')
            with gzip.open(tmp,'wt',encoding='utf-8') as out:
                total=0
                for page_index,page,url in discover_pages(query,period,http):
                    total+=len(page.get('features',[]))
                    if total>MAX_STAC_ITEMS_PER_QUARTER:raise RuntimeError('Exceso de Items STAC por trimestre')
                    out.write(json.dumps(page,ensure_ascii=False)+'\n')
                    if tmp.stat().st_size>512*1024**2:raise RuntimeError('Archivo de catálogo supera límite 512 MiB comprimidos')
                    if page_index%10==0:print(period,'páginas',page_index+1,'Items recuperados',total,flush=True)
            tmp.replace(provider_path)
            cached={'complete':True,'sha256':digest_file(provider_path),'query':query,'obtained_at':utc()}
            cache_meta.write_text(json.dumps(cached))
            s3.upload_file(str(provider_path),CONFIG['buckets']['operations'],cache_key,Config=TRANSFER)
            put_json(CONFIG['buckets']['operations'],cache_meta_key,cached)
        else:print(period,'CATALOGO_REUTILIZADO',flush=True)

        # A complete local snapshot must also exist remotely before workers consume it.
        try:s3.head_object(Bucket=CONFIG['buckets']['operations'],Key=cache_key)
        except ClientError as exc:
            if exc.response['Error']['Code'] not in ['404','NoSuchKey','NotFound']:raise
            s3.upload_file(str(provider_path),CONFIG['buckets']['operations'],cache_key,Config=TRANSFER)
            put_json(CONFIG['buckets']['operations'],cache_meta_key,cached)
        return {'period':period,'provider_path':str(provider_path),'query_id':query_id,'cached':cached,'catalog_key':cache_key,'catalog_sha256':cached['sha256'],'catalog_seconds':round(time.perf_counter()-start,3),'catalog_cache_hit':bool(cache_valid)}
    finally:http.close()

if not RESUME_PLAN_URI:
    planning_start=time.perf_counter()
    import geopandas as gpd
    from shapely.geometry import shape, mapping
    from shapely.ops import unary_union, transform
    from pyproj import Transformer
    import subprocess, calendar
    if PLANNING_ERRORS: raise RuntimeError('Fuentes anteriores tienen incidencias; corregir antes de consultar Sentinel.')
    districts=None
    # Solo se descarga el archivo distrital para derivar el AOI; se preservan sus bytes.
    if PERU_AOI_INPUT:
        if urlparse(PERU_AOI_INPUT).scheme=='s3':
            b,k=s3_parts(PERU_AOI_INPUT); obj=get_json(b,k)
        else: obj=json.loads(Path(PERU_AOI_INPUT).read_text(encoding='utf-8'))
        if obj.get('type')=='FeatureCollection': national=unary_union([shape(f['geometry']) for f in obj['features']])
        elif obj.get('type')=='Feature': national=shape(obj['geometry'])
        else: national=shape(obj)
    else:
        extractor=shutil.which('unrar') or shutil.which('bsdtar')
        if not extractor: raise RuntimeError('Falta unrar/bsdtar remoto. Instálelo vía bootstrap o aporte PERU_AOI_INPUT derivado del INEI.')
        rar=CACHE/'Distrito.rar'
        if INEI_DISTRICT_INPUT_URI:
            bucket,key=s3_parts(INEI_DISTRICT_INPUT_URI)
            h=s3.head_object(Bucket=bucket,Key=key)
            if not 0<h['ContentLength']<=128*1024**2:raise ValueError('RAR fuera del límite de planificación')
            extra={'VersionId':h['VersionId']} if h.get('VersionId') else {}
            s3.download_file(bucket,key,str(rar),ExtraArgs=extra)
            print('INEI_DISTRITAL_REUTILIZADO',INEI_DISTRICT_INPUT_URI)
        else:
            with HTTP.get('https://ide.inei.gob.pe/files/Distrito.rar',stream=True,timeout=(15,120)) as response:
                response.raise_for_status()
                if urlparse(response.url).scheme!='https':raise ValueError('Redirección insegura')
                with rar.open('wb') as out:
                    count=0
                    for block in response.iter_content(8*1024*1024):
                        count+=len(block)
                        if count>128*1024**2:raise ValueError('RAR excede el límite')
                        out.write(block)
        extract=CACHE/'inei_distrito'; extract.mkdir(exist_ok=True)
        args=[extractor,'x','-o+',str(rar),str(extract)+'/'] if Path(extractor).name=='unrar' else [extractor,'-xf',str(rar),'-C',str(extract)]
        subprocess.run(args,check=True,capture_output=True,timeout=120)
        packages=list(extract.rglob('*.gpkg')) or list(extract.rglob('*.shp'))
        if len(packages)!=1: raise RuntimeError('Se requiere identificar una única capa distrital GPKG/SHP; revise el RAR.')
        import pyogrio
        layers=pyogrio.list_layers(packages[0])
        if len(layers)>1 and not INEI_DISTRICT_LAYER: raise RuntimeError('GPKG con varias capas: configure INEI_DISTRICT_LAYER.')
        districts=gpd.read_file(packages[0],engine='pyogrio',layer=INEI_DISTRICT_LAYER)
        if districts.crs is None or districts.empty: raise ValueError('Capa INEI sin CRS o vacía')
        districts=districts.to_crs(4326)
        if (~districts.geometry.is_valid).any(): raise ValueError('Geometrías INEI inválidas: corregir AOI y documentar antes de continuar')
        national=unary_union(list(districts.geometry))
        # Ingerir exactamente el RAR usado para construir el AOI.
        for row in INVENTORY:
            if row['source']=='inei' and row.get('level')=='distrito': row['locator']=str(rar); row['delivery_method']='S3_original_preservado' if INEI_DISTRICT_INPUT_URI else 'HTTPS_descargado_durante_planificacion'; row['obtained_at']=utc()
    if national.is_empty or not national.is_valid: raise ValueError('AOI nacional vacío o inválido')
    west,south,east,north=national.bounds
    if not (-83<west<-79 and -20<south<-16 and -71<east<-67 and -2<north<2): raise ValueError('AOI no tiene extensión nacional esperada de Perú; verificar EPSG:4326')
    if districts is not None:
        if len(districts)!=SPEC['geography']['expected_districts']: raise RuntimeError('Catálogo INEI distinto del contrato; conciliar versión antes de continuar')
    else:
        # Se necesita capa distrital para reportar áreas sin huella por distrito.
        districts=None
    aoi_path=local_json('peru_aoi_inei.geojson',{'type':'Feature','properties':{'source':'INEI','crs':'EPSG:4326','derived':True},'geometry':mapping(national)})
    INVENTORY.append(save_planning_metadata('inei','peru_aoi_inei.geojson',json.loads(Path(aoi_path).read_text()),derived=True,provenance='https://ide.inei.gob.pe',license='condiciones_INEI',metadata_origin='union_geometrica_derivada'))

    for row in INVENTORY:
        if row['source']=='inei' and row.get('level')=='distrito' and urlparse(row['locator']).scheme=='':
            staged=OP_PREFIX+'/planning_inputs/Distrito.rar'
            s3.upload_file(row['locator'],CONFIG['buckets']['operations'],staged)
            row['locator']=f"s3://{CONFIG['buckets']['operations']}/{staged}"
            row['expected_sha256']=digest_file(rar)
            row['expected_bytes']=rar.stat().st_size
    base=f'https://earth-search.aws.element84.com/v1/collections/{STAC_COLLECTION}'
    collection=json_url(base)
    INVENTORY.append(save_planning_metadata('sentinel','collection.json',collection,publisher='Element84/Copernicus/ESA',license='Copernicus_Sentinel_legal_notice',provenance=base))
    catalog_cache=Path(CATALOG_CACHE_ROOT);catalog_cache.mkdir(parents=True,exist_ok=True)
    from concurrent.futures import ThreadPoolExecutor
    selection_cfg={'selector_source':SELECTION_SOURCE,'aoi_wkb':national.wkb_hex,
                   'bands':SENTINEL['bands'],'max_scene_cloud_pct':SENTINEL['max_scene_cloud_pct'],
                   'required_fraction':SENTINEL['required_footprint_fraction'],'operations_bucket':CONFIG['buckets']['operations']}
    selection_bc=sc.broadcast(selection_cfg)
    print('SELECCION_DISTRIBUIDA_SPARK; UI:',sc.uiWebUrl,'consultas concurrentes:',CATALOG_CONCURRENCY)
    try:
        for offset in range(0,len(PERIODS),SELECTION_PARTITIONS):
            batch_periods=PERIODS[offset:offset+SELECTION_PARTITIONS]
            with ThreadPoolExecutor(max_workers=min(CATALOG_CONCURRENCY,len(batch_periods))) as pool:
                jobs=list(pool.map(prepare_catalog,batch_periods))
            for job in jobs:
                fingerprint={'catalog_sha256':job['catalog_sha256'],'aoi_sha256':hashlib.sha256(national.wkb).hexdigest(),
                             'selector_sha256':hashlib.sha256(SELECTION_SOURCE.encode()).hexdigest(),'policy':{k:selection_cfg[k] for k in ['bands','max_scene_cloud_pct','required_fraction']},'revision':'4.0'}
                job['selection_id']=hashlib.sha256(json.dumps(fingerprint,sort_keys=True).encode()).hexdigest()
                period=job['period']
                INVENTORY.append(archive_entry('sentinel',f'stac_responses_{period}.jsonl.gz',Path(job['provider_path']),period))
                INVENTORY.append(save_planning_metadata('sentinel',f'query_{period}.json',job['cached'],period,derived=True))
            remote_jobs=[{k:job[k] for k in ['period','catalog_key','catalog_sha256','selection_id']} for job in jobs]
            rdd=sc.parallelize(remote_jobs,len(remote_jobs)).mapPartitions(selection_worker_factory(selection_bc))
            reports=collect_with_progress(sc,rdd,'seleccion_'+str(offset//SELECTION_PARTITIONS+1))
            by_period={r['period']:r for r in reports}
            if set(by_period)!=set(batch_periods):raise RuntimeError('Spark no devolvió todos los trimestres del lote')
            for job in jobs:
                report=by_period[job['period']];period=job['period']
                if report['status']!='ok':raise RuntimeError(f"Selección remota {period}: {report.get('error_type')}: {report.get('reason')}")
                year=int(period[:4]);quarter=int(period[-1]);month=3*(quarter-1)+1;end_month=month+2
                inventory_start=time.perf_counter()
                selected_path=CACHE/f'selected_items_{period}.jsonl.gz'
                s3.download_file(CONFIG['buckets']['operations'],report['selected_key'],str(selected_path))
                if digest_file(selected_path)!=report['selected_sha256']:raise ValueError('SHA selección remota no coincide')
                with gzip.open(selected_path,'rt',encoding='utf-8') as selected_file:selected_items=[json.loads(line) for line in selected_file if line.strip()]
                selected_ids=report['selected_ids'];selected_id_set=set(selected_ids)
                if {item['id'] for item in selected_items}!=selected_id_set:raise ValueError('IDs de selección no coinciden con cache')
                coverage={k:report[k] for k in ['candidate_footprint_coverage','national_footprint_coverage','repair_swaps','coverage_precision_m','eligible_candidate_count','missing_fraction']}
                items={item['id']:None for item in selected_items}  # no se reconstruyen miles de candidatas en el driver
                non_public_band_items=set(report['non_public_band_item_ids']);metadata_resolutions=[]
                inverse=Transformer.from_crs(4326,6933,always_xy=True).transform
                uncovered_projected=transform(inverse,shape(report['uncovered']['geometry']))
                project=inverse
                selected={(str(t),int(m)):None for t,m in report['months_selected']}
                affected=[]
                if districts is not None and coverage['missing_fraction']>1-SENTINEL['required_footprint_fraction']:
                    ubigeo_cols=[c for c in districts.columns if str(c).lower() in ['ubigeo','ubigeo_distrito','cod_distrito','ccddccppccdi']]
                    if not ubigeo_cols:raise RuntimeError('No se reconoce campo UBIGEO en INEI para diagnóstico distrital')
                    for _,row in districts.iterrows():
                        geom=transform(project,row.geometry);missing=geom.intersection(uncovered_projected).area
                        if geom.area>0 and missing/geom.area>1e-6:
                            affected.append({'ubigeo':str(row[ubigeo_cols[0]]),'uncovered_fraction':missing/geom.area})
                selection={'period':period,'scope':'Peru_nacional','collection':STAC_COLLECTION,
                           'candidate_count':report['candidate_count'],'selected_count':len(selected_items),
                           'selected_ids':selected_ids,**coverage,
                           'minimum_required':SENTINEL['required_footprint_fraction'],
                           'method':CONFIG['ingestion_implementation']['selection'],
                           'coverage_crs':'EPSG:6933','affected_districts':affected if districts is not None else None,
                           'valid_pixels_not_yet_assessed':True,'asset_access_policy':'public_https_bands_and_documented_metadata_proxy',
                           'non_public_band_item_count':len(non_public_band_items),'non_public_band_item_ids':sorted(non_public_band_items),
                           'monthly_missing':[{'mgrs':tile,'month':m} for tile in sorted({k[0] for k in selected}) for m in range(month,end_month+1) if (tile,m) not in selected]}
                selection['execution']={'worker_host':report['worker_host'],'worker_python':report['worker_python'],'cache_hit':report.get('cache_hit'),'seconds':report['seconds'],'phase_seconds':report.get('phase_seconds')}
                COVERAGE_REPORTS.append(selection)
                INVENTORY.append(save_planning_metadata('sentinel',f'selection_{period}.json',selection,period,derived=True))
                inverse=Transformer.from_crs(6933,4326,always_xy=True).transform
                INVENTORY.append(save_planning_metadata('sentinel',f'uncovered_{period}.geojson',{'type':'Feature','properties':{'period':period,'derived':True},'geometry':mapping(transform(inverse,uncovered_projected))},period,derived=True))
                INVENTORY.append(archive_entry('sentinel',selected_path.name,selected_path,period))
                print(period,'worker',report['worker_host'],'SELECCION_REUTILIZADA' if report.get('cache_hit') else 'SELECCION_SPARK','fases',report.get('phase_seconds'),flush=True)
                print(period,'candidatas',report['candidate_count'],'seleccionadas',len(selected_items),
                      'cobertura candidatas %',round(100*coverage['candidate_footprint_coverage'],6),
                      'seleccionadas %',round(100*coverage['national_footprint_coverage'],6),flush=True)
                # Include all selected assets even for a failed quarter; inventory is diagnostic, not certified.
                for item in selected_items:
                    item_url='https://earth-search.aws.element84.com/v1/collections/'+item.get('collection',STAC_COLLECTION)+'/items/'+item['id'];acquisition=item['properties']['datetime']
                    for asset_key,asset in item['assets'].items():
                        is_band=asset_key in SENTINEL['bands']
                        suffix=Path(urlparse(asset['href']).path).suffix.lower()
                        if not is_band and suffix not in ['.json','.xml']:continue
                        locator,resolution=resolve_sentinel_asset(item,asset_key,asset,is_band)
                        if resolution['method']!='catalog_https':
                            metadata_resolutions.append({'scene_id':item['id'],'asset_key':asset_key,**resolution})
                        INVENTORY.append(entry('sentinel',locator,Path(urlparse(asset['href']).path).name,
                            'data' if is_band else 'metadata',period=acquisition,publisher='Copernicus/ESA; Element84',
                            license='Copernicus_Sentinel_legal_notice',provenance=item_url,
                            scene_id=item['id'],asset_key=asset_key,analysis_period=period,
                            acquisition_month=acquisition[:7],collection=item.get('collection',STAC_COLLECTION),catalog_query_collection=STAC_COLLECTION,
                            processing_baseline=item['properties'].get('s2:processing_baseline'),
                            boa_offset_applied=item['properties'].get('earthsearch:boa_offset_applied'),
                            asset_metadata=asset,catalog_asset_locator=asset['href'],asset_resolution=resolution,
                            expected_bytes=asset.get('file:size') if locator==asset['href'] else None,
                            declared_checksum=asset.get('file:checksum') if locator==asset['href'] else None))
                INVENTORY.append(save_planning_metadata('sentinel',f'asset_resolution_{period}.json',metadata_resolutions,period,derived=True))
                print(period,'metadatos resueltos por proxy público:',len(metadata_resolutions),flush=True)
                if coverage['candidate_footprint_coverage']<SENTINEL['required_footprint_fraction']:
                    raise RuntimeError('Cobertura de candidatas con cuatro bandas HTTPS públicas insuficiente: revisar catálogo/AOI; no ampliar trimestres ni bajar umbral automáticamente')
                if coverage['national_footprint_coverage']<SENTINEL['required_footprint_fraction']:
                    raise RuntimeError('Selección mensual no alcanza cobertura; revisar uncovered y regla sin declarar éxito')
                PHASE_METRICS.append({'phase':'quarter_planning','period':period,'catalog_seconds':job['catalog_seconds'],'selection_seconds':report['seconds'],'selection_cache_hit':report.get('cache_hit'),'worker_host':report['worker_host'],'geometry_phase_seconds':report.get('phase_seconds'), 'driver_inventory_seconds':round(time.perf_counter()-inventory_start,3)})
                put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/coverage_checkpoint.json',COVERAGE_REPORTS)
                put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/phase_metrics.json',PHASE_METRICS)
    except Exception as exc:
        planning_error('sentinel',locals().get('period','batch'),exc)
        put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/coverage_checkpoint.json',COVERAGE_REPORTS)
        raise RuntimeError(f'PLANIFICACION_DISTRIBUIDA_DETENIDA: {type(exc).__name__}: {exc}. Se conservan catálogos y selecciones completas.') from exc
    finally:selection_bc.unpersist()
    INVENTORY.extend(EXTRA_ENTRIES)
    PHASE_METRICS.append({'phase':'planning_wall_time','seconds':round(time.perf_counter()-planning_start,3),'quarters':len(COVERAGE_REPORTS)})
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/phase_metrics.json',PHASE_METRICS)
print('Inventario nacional:',len(INVENTORY),'objetos; incidencias:',len(PLANNING_ERRORS))


## 7. Preflight temprano, tamaños y congelación
Antes de solicitudes masivas comprueba incidencias, seis fuentes, 29 reportes, cobertura y bandas. Después obtiene tamaños exactos con ocho workers HTTP/HEAD. No utiliza el tamaño del piloto como tamaño real de todos los archivos ni asigna 8 GiB a cada tamaño desconocido. Los metadatos recién escritos no requieren otro HEAD de preflight.

El techo operativo de esta revisión es 8192 GiB. Verifique `TAMAÑO_REAL_DEL_PLAN_GiB` antes de ejecutar sección 9. Este límite no implica un costo aprobado externamente ni cambia los 29 trimestres. Un proveedor sin tamaño verificable sigue siendo una incidencia explícita.

In [ ]:
preflight_start=time.perf_counter()
REQUIRED={'ookla','osiptel_cobertura','osiptel_estaciones','inei','worldpop','sentinel'}
if PLANNING_ERRORS:raise RuntimeError('PLAN_INCOMPLETO: revisar planning_errors.json antes de hacer HEAD.')
if len(COVERAGE_REPORTS)!=len(PERIODS) or {c['period'] for c in COVERAGE_REPORTS}!=set(PERIODS):raise RuntimeError('Faltan trimestres Sentinel')
if any(c['national_footprint_coverage']<SENTINEL['required_footprint_fraction'] for c in COVERAGE_REPORTS):raise RuntimeError('Cobertura insuficiente')
if not INVENTORY or len(INVENTORY)>MAX_INVENTORY_FILES:raise ValueError('Inventario vacío o excesivo')
missing=REQUIRED-{r['source'] for r in INVENTORY if r['kind']=='data'}
if missing:raise RuntimeError('Fuentes de datos ausentes: '+str(missing))
seen=set();scenes={}
for i,row in enumerate(INVENTORY):
    for field in ['source','locator','original_name','kind','publisher','license','provenance_url']:
        if not row.get(field):raise ValueError(f'Fila {i}: falta {field}')
    if row['source'] not in REQUIRED:raise ValueError('Fuente desconocida')
    name=row['original_name']
    if '/' in name or chr(92) in name or name in ['.','..'] or not name.strip():raise ValueError('Nombre inseguro')
    if row['kind'] not in ['data','metadata']:raise ValueError('Kind inválido')
    if row.get('derived') and row.get('target_zone')!='operations':raise ValueError('Derivado fuera de Operations')
    pair=(row['source'],row['locator'],name)
    if pair in seen:raise ValueError('Entrada repetida: reiniciar kernel, no reejecutar inventarios')
    seen.add(pair)
    if row['source']=='sentinel' and row['kind']=='data':scenes.setdefault(row['scene_id'],set()).add(row['asset_key'])
if any(not set(SENTINEL['bands'])<=bands for bands in scenes.values()):raise ValueError('Escena sin bandas completas')
if len([r for r in INVENTORY if r['source']=='inei' and r['kind']=='data'])!=3:raise ValueError('INEI requiere tres capas')

def inspect_entry(pair):
    i,row=pair;scheme=urlparse(row['locator']).scheme;updates={}
    if row.get('staged_in_current_run') and not RESUME_PLAN_URI:
        size=row['expected_bytes']  # objeto recién escrito y tamaño/hash del cuerpo conocido
    elif scheme=='s3':
        b,k=s3_parts(row['locator']);client=PUBLIC_S3 if row.get('anonymous') else s3
        args={'Bucket':b,'Key':k}
        if row.get('input_version_id'):args['VersionId']=row['input_version_id']
        h=client.head_object(**args);size=h['ContentLength']
        updates={'input_version_id':h.get('VersionId'),'preflight_etag':h.get('ETag')}
    elif scheme=='https':
        # Independent requests per worker, no shared mutable Session.
        size=None
        for attempt in range(4):
            try:
                with requests.head(row['locator'],allow_redirects=True,timeout=(15,60)) as r:
                    r.raise_for_status()
                    if urlparse(r.url).scheme!='https':raise ValueError('Redirección no HTTPS')
                    if r.headers.get('Content-Length'):size=int(r.headers['Content-Length'])
                    updates={'preflight_etag':r.headers.get('ETag')}
                if size is None:
                    with requests.get(row['locator'],headers={'Range':'bytes=0-0','Accept-Encoding':'identity'},stream=True,timeout=(15,60)) as r:
                        r.raise_for_status();content_range=r.headers.get('Content-Range','')
                        if '/' in content_range and content_range.rsplit('/',1)[1].isdigit():size=int(content_range.rsplit('/',1)[1])
                        elif r.status_code==200 and r.headers.get('Content-Length'):size=int(r.headers['Content-Length'])
                break
            except requests.RequestException:
                if attempt==3:raise
                time.sleep(2*(attempt+1))
        if size is None:raise ValueError(f'Fila {i}: proveedor no declara tamaño; no se inventa una cota por escena')
    elif scheme=='':
        path=Path(row['locator'])
        if not path.is_file():raise FileNotFoundError(str(path))
        size=path.stat().st_size
    else:raise ValueError('Solo HTTPS, S3 o ruta remota')
    if not 0<size<=MAX_FILE_GIB*1024**3:raise ValueError(f'Fila {i}: tamaño fuera del límite')
    if row.get('expected_bytes') is not None and size!=row['expected_bytes']:raise ValueError(f'Fila {i}: bytes distintos del catálogo')
    return i,size,updates

estimate=0
with ThreadPoolExecutor(max_workers=PREFLIGHT_WORKERS) as pool:
    for j,(i,size,updates) in enumerate(pool.map(inspect_entry,enumerate(INVENTORY)),1):
        INVENTORY[i].update(planned_bytes=size,**updates);estimate+=size
        if j%1000==0:print('PRECHECK',j,'/',len(INVENTORY),'GiB',round(estimate/1024**3,2),flush=True)
        if estimate>MAX_TOTAL_GIB*1024**3:raise ValueError('Plan supera MAX_TOTAL_GIB: detener y revisar volumen; no se transfiere')
print('TAMAÑO_REAL_DEL_PLAN_GiB',round(estimate/1024**3,3),'límite_GiB',MAX_TOTAL_GIB)
for i,row in enumerate(INVENTORY):
    if urlparse(row['locator']).scheme=='':
        local=Path(row['locator']);sha=digest_file(local)
        key=f"{OP_PREFIX}/planning_inputs/{i:06d}/{row['original_name']}"
        s3.upload_file(str(local),CONFIG['buckets']['operations'],key,Config=TRANSFER)
        remote_sha,remote_size=digest_object(CONFIG['buckets']['operations'],key)
        if (remote_sha,remote_size)!=(sha,local.stat().st_size):raise RuntimeError('Entrada congelada inválida')
        row.update(locator=f"s3://{CONFIG['buckets']['operations']}/{key}",expected_sha256=sha)
PLAN={'schema_version':2,'spec_version':'1.0.0','ingestion_revision':INGESTION_REVISION,
      'run_id':RUN_ID,'created_at':utc(),'config':CONFIG,'entries':INVENTORY,
      'estimated_bytes_upper_bound':estimate,'planning_errors':PLANNING_ERRORS,
      'coverage_reports':COVERAGE_REPORTS,'resumed_from':RESUME_PLAN_URI}
PLAN_URI=f"s3://{CONFIG['buckets']['operations']}/{OP_PREFIX}/plan.json"
put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/plan.json',PLAN)
print('PLAN_COMPLETO_LISTO_PARA_TRANSFERIR',len(INVENTORY),'objetos')
print('PLAN_URI',PLAN_URI)

PHASE_METRICS.append({'phase':'preflight','seconds':round(time.perf_counter()-preflight_start,3),'objects':len(INVENTORY)})
put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/phase_metrics.json',PHASE_METRICS)


## 8. Transferencia y validación de archivos
Las funciones se inicializan aisladamente en cada executor. Los inputs deben ser HTTPS o S3 después del preflight; nunca rutas locales del primary. SHA-256 remoto completo permanece obligatorio.


In [ ]:
TRANSFER_SOURCE = r"""
def reserve(size):
    required=(size if size is not None else MAX_FILE_GIB*1024**3)+MIN_FREE_GIB*1024**3
    if shutil.disk_usage(CACHE).free<required: raise RuntimeError('Disco insuficiente: cambie CACHE_ROOT/límites o adjunte volumen.')
def materialize(row):
    locator=row['locator']; scheme=urlparse(locator).scheme
    if scheme=='': return Path(locator),False,{}
    path=CACHE/'payload.download'; reserve(row.get('planned_bytes'))
    for attempt in range(1,4):
        try:
            info={}
            if scheme=='s3':
                bucket,key=s3_parts(locator); client=PUBLIC_S3 if row.get('anonymous') else s3
                args={'Bucket':bucket,'Key':key}
                if row.get('input_version_id'): args['VersionId']=row['input_version_id']
                elif row.get('preflight_etag'): args['IfMatch']=row['preflight_etag']
                response=client.get_object(**args)
                with closing(response['Body']) as body, path.open('wb') as out:
                    count=0
                    for block in iter(lambda:body.read(8*1024*1024),b''):
                        count+=len(block)
                        if count>MAX_FILE_GIB*1024**3: raise ValueError('Archivo supera MAX_FILE_GIB')
                        out.write(block)
                info={'etag':response.get('ETag'),'version_id':response.get('VersionId'),'last_modified':str(response.get('LastModified'))}
            else:
                with HTTP.get(locator,headers={'If-Match':row['preflight_etag']} if row.get('preflight_etag') else {},stream=True,timeout=(15,120)) as response:
                    response.raise_for_status()
                    if row.get('preflight_etag') and response.headers.get('ETag') and response.headers['ETag']!=row['preflight_etag']: raise ValueError('Objeto HTTPS cambió desde el preflight')
                    if urlparse(response.url).scheme!='https': raise ValueError('Redirección no HTTPS')
                    info={'effective_url':response.url,'etag':response.headers.get('ETag'),'last_modified':response.headers.get('Last-Modified'),'content_type':response.headers.get('Content-Type')}
                    count=0
                    with path.open('wb') as out:
                        for block in response.iter_content(chunk_size=8*1024*1024):
                            count+=len(block)
                            if count>MAX_FILE_GIB*1024**3: raise ValueError('Archivo supera MAX_FILE_GIB')
                            out.write(block)
            return path,True,info
        except (requests.RequestException,ClientError,OSError) as exc:
            path.unlink(missing_ok=True)
            logger.warning('Reintento %d: %s',attempt,type(exc).__name__)
            if attempt==3: raise
            time.sleep(attempt*2)
def validate_file(path,name,kind):
    size=path.stat().st_size
    if not 0<size<=MAX_FILE_GIB*1024**3: raise ValueError('Archivo vacío o demasiado grande')
    with path.open('rb') as f: first=f.read(512); f.seek(max(0,size-4)); last=f.read(4)
    suffix=Path(name).suffix.lower()
    if suffix=='.parquet' and (first[:4]!=b'PAR1' or last!=b'PAR1'): raise ValueError('Firma Parquet inválida')
    if suffix=='.xlsx' and not first.startswith(b'PK'): raise ValueError('Firma XLSX inválida')
    if suffix in ['.tif','.tiff'] and first[:4] not in [b'II*\x00',b'MM\x00*',b'II+\x00',b'MM\x00+']: raise ValueError('Firma TIFF inválida')
    if suffix=='.rar' and not first.startswith(b'Rar!'): raise ValueError('Firma RAR inválida')
    if suffix=='.csv' and (b'<html' in first.lower() or b'<!doctype html' in first.lower()): raise ValueError('CSV es una página HTML')
    if suffix=='.gz':
        with gzip.open(path,'rt',encoding='utf-8') as records:
            for line in records:
                if line.strip():json.loads(line)
    if suffix=='.xml':
        import xml.etree.ElementTree as ET
        ET.parse(path)
    if suffix in ['.json','.geojson']:
        with path.open(encoding='utf-8') as f: json.load(f)
    return size
def ingest_one(row,index):
    start=time.perf_counter(); result={'index':index,'run_id':RUN_ID,'entry':row,'started_at':utc(),'status':'failed'}
    path=None; temporary=False
    try:
        path,temporary,source_headers=materialize(row)
        size=validate_file(path,row['original_name'],row['kind'])
        expected=row.get('expected_bytes') or row.get('planned_bytes')
        if expected is not None and size!=expected: raise ValueError('Bytes recibidos no coinciden con inventario')
        sha=digest_file(path)
        if row.get('expected_sha256') and sha!=row['expected_sha256']: raise ValueError('SHA-256 no coincide con proveedor')
        key=f"{CONFIG['bronze_root']}/{row['source']}/snapshot_id={sha}/{row['kind']}/{row['original_name']}"
        zone=row.get('target_zone','bronze')
        if zone not in ['bronze','operations']: raise ValueError('Zona de destino no admitida')
        if zone=='operations': key=f"{OP_PREFIX}/derived/{row['source']}/snapshot_id={sha}/{row['original_name']}"
        bucket=CONFIG['buckets'][zone];exists=None
        try: exists=s3.head_object(Bucket=bucket,Key=key)
        except ClientError as exc:
            if exc.response['Error']['Code'] not in ['404','NoSuchKey','NotFound']: raise
        if exists:
            remote_sha,remote_size=digest_object(bucket,key,exists.get('VersionId'))
            if (remote_sha,remote_size)!=(sha,size): raise RuntimeError('Conflicto en ruta por contenido; no se sobrescribe')
            status='already_verified'
        else:
            s3.upload_file(str(path),bucket,key,ExtraArgs={'Metadata':{'sha256':sha,'source':row['source'],'run-id':RUN_ID}},Config=TRANSFER)
            exists=s3.head_object(Bucket=bucket,Key=key)
            remote_sha,remote_size=digest_object(bucket,key,exists.get('VersionId'))
            if (remote_sha,remote_size)!=(sha,size): raise RuntimeError('Verificación remota falló')
            status='uploaded_verified'
        result.update(target_zone=zone,status=status,bytes=size,sha256=sha,bucket=bucket,key=key,uri=f's3://{bucket}/{key}',version_id=exists.get('VersionId'),sse=exists.get('ServerSideEncryption'),kms_key_id=exists.get('SSEKMSKeyId'),source_headers=source_headers,integrity='full_object_sha256',ingested_at=utc())
    except Exception as exc:
        # No registrar URLs firmadas ni tokens en mensajes de excepción.
        result.update(error_type=type(exc).__name__)
        if isinstance(exc,(ValueError,RuntimeError,AttributeError,NameError,TypeError)): result['reason']=str(exc)[:400]
        if isinstance(exc,ClientError): result['aws_error_code']=exc.response['Error']['Code']
        logger.error('Fila %d falló (%s)',index,type(exc).__name__)
    finally:
        if temporary and path is not None: path.unlink(missing_ok=True)
        (CACHE/'payload.download').unlink(missing_ok=True)
        result['finished_at']=utc(); result['seconds']=round(time.perf_counter()-start,3)
    return result"""
# El worker inicializa este código con clientes y directorio propios.


## 9. Transferencias distribuidas y evidencia
Primero prueba S3 y disco desde tareas remotas. Ejecuta hasta seis particiones con un archivo simultáneo por host; cada tarea usa un directorio y clientes propios. Recibos se escriben después de cada archivo. Los checkpoints del driver se actualizan por lote. Spark puede reintentar tareas: destinos por SHA y verificación completa conservan idempotencia lógica; no se promete una única solicitud física. No active speculation.


In [ ]:
RESULTS=[]
if not EXECUTE_INGESTION:
    print('PLAN_ONLY: plan completo persistido; no se transfirieron datasets.')
else:
    if PLANNING_ERRORS:raise RuntimeError('Se prohíbe transferir un plan con incidencias')
    if any(urlparse(r['locator']).scheme not in ['https','s3'] for r in INVENTORY):raise ValueError('Plan no portable: congele todos los archivos locales en sección 7')
    access_cfg={'op_prefix':OP_PREFIX,'buckets':CONFIG['buckets'],'max_file_gib':MAX_FILE_GIB,'min_free_gib':MIN_FREE_GIB}
    probes=collect_with_progress(sc,sc.parallelize(range(TRANSFER_PARTITIONS),TRANSFER_PARTITIONS).mapPartitions(worker_access_probe_factory(access_cfg)),'worker_s3_preflight')
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/worker_s3_preflight.json',probes)
    print('WORKERS_S3_Y_DISCO_OK',sorted({p['host'] for p in probes}),flush=True)
    settings={'transfer_source':TRANSFER_SOURCE,'run_id':RUN_ID,'config':CONFIG,'op_prefix':OP_PREFIX,
              'operations_bucket':CONFIG['buckets']['operations'],'max_file_gib':MAX_FILE_GIB,'min_free_gib':MIN_FREE_GIB}
    transfer_bc=sc.broadcast(settings);transfer_start=time.perf_counter()
    try:
        for offset in range(0,len(INVENTORY),TRANSFER_BATCH_SIZE):
            indexed=list(enumerate(INVENTORY[offset:offset+TRANSFER_BATCH_SIZE],start=offset))
            batch=sc.parallelize(indexed,min(TRANSFER_PARTITIONS,len(indexed))).mapPartitions(transfer_worker_factory(transfer_bc))
            received=collect_with_progress(sc,batch,'transferencia_'+str(offset//TRANSFER_BATCH_SIZE+1))
            if {r['index'] for r in received}!={i for i,_ in indexed}:raise RuntimeError('Recibos del lote incompletos')
            RESULTS.extend(sorted(received,key=lambda r:r['index']))
            put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/checkpoint.json',{'run_id':RUN_ID,'processed':len(RESULTS),'expected':len(INVENTORY),'updated_at':utc()})
            for result in received:logger.info('Fila=%d worker=%s status=%s bytes=%s segundos=%.3f',result['index'],result['worker_host'],result['status'],result.get('bytes'),result['seconds'])
            for handler in logger.handlers:handler.flush()
            s3.upload_file(str(LOG_PATH),CONFIG['buckets']['operations'],OP_PREFIX+'/ingestion.log')
            print('TRANSFERIDOS',len(RESULTS),'/',len(INVENTORY),'hosts',sorted({r['worker_host'] for r in RESULTS}),'fallos',sum(r['status']=='failed' for r in RESULTS),flush=True)
        PHASE_METRICS.append({'phase':'distributed_transfer','seconds':round(time.perf_counter()-transfer_start,3),'objects':len(RESULTS),'worker_hosts':sorted({r['worker_host'] for r in RESULTS})})
        put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/phase_metrics.json',PHASE_METRICS)
    finally:transfer_bc.unpersist()
print('Procesados:',len(RESULTS))


## 10. Conciliación y bundle
Éxito exige todas las entradas, cero incidencias y cobertura de los 29 trimestres. Un fallo produce _FAILED y no publica bundle completo. El bundle diferencia originales de artefactos derivados.

In [ ]:
if EXECUTE_INGESTION:
    complete=not PLANNING_ERRORS and len(COVERAGE_REPORTS)==len(PERIODS) and all(c['national_footprint_coverage']>=SENTINEL['required_footprint_fraction'] for c in COVERAGE_REPORTS) and len(RESULTS)==len(INVENTORY) and all(r['status'] in ['uploaded_verified','already_verified'] for r in RESULTS)
    summary={'schema_version':2,'spec_version':'1.0.0','run_id':RUN_ID,'finished_at':utc(),'complete':complete,'expected_objects':len(INVENTORY),'processed_objects':len(RESULTS),'failed_objects':sum(r['status']=='failed' for r in RESULTS),'verified_bytes':sum(r.get('bytes',0) for r in RESULTS if r['status']!='failed'),'scope':CONFIG['ingestion_scope'],'sentinel_scope':'Peru_nacional','sentinel_periods':PERIODS,'planning_errors':PLANNING_ERRORS,'verified_objects':sum(r['status']!='failed' for r in RESULTS),'sources':{s:sum(r['entry']['source']==s and r['status']!='failed' for r in RESULTS) for s in sorted(REQUIRED)}}
    body=''.join(json.dumps(r,ensure_ascii=False,default=str)+'\n' for r in RESULTS).encode()
    manifest_key=OP_PREFIX+'/manifest.jsonl'
    s3.put_object(Bucket=CONFIG['buckets']['operations'],Key=manifest_key,Body=body,ContentType='application/x-ndjson')
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/summary.json',summary)
    if complete:
        bundle={'schema_version':2,'spec_version':'1.0.0','run_id':RUN_ID,'config':CONFIG,'summary':summary,'objects':RESULTS,'original_objects':[r for r in RESULTS if r.get('target_zone')=='bronze'],'derived_artifacts':[r for r in RESULTS if r.get('target_zone')=='operations'],'plan_uri':PLAN_URI,'manifest_uri':f"s3://{CONFIG['buckets']['operations']}/{manifest_key}"}
        bundle_key=f'bundles/run_id={RUN_ID}/bundle.json'
        put_json(CONFIG['buckets']['bronze'],bundle_key,bundle)
        summary['bundle_uri']=f"s3://{CONFIG['buckets']['bronze']}/{bundle_key}"
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/summary.json',summary)
    if not complete: put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/_FAILED.json',summary)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
    if not complete: raise RuntimeError('INGESTA_PARCIAL: revise recibos fallidos; corrija entradas/permisos/red y reejecute. No iniciar Silver.')

## 11. Verificación Spark y marcador final
Solo después de leer y conciliar el manifiesto se publica _SUCCESS. Si esta prueba falla, no hay marcador de éxito aunque los originales estén transferidos.

In [ ]:
if EXECUTE_INGESTION:
    if 'spark' not in globals(): raise RuntimeError('Se requiere sesión Spark para validar lectura distribuida del manifiesto.')
    manifest=spark.read.json(f"s3://{CONFIG['buckets']['operations']}/{OP_PREFIX}/manifest.jsonl")
    assert manifest.count()==len(INVENTORY)
    assert manifest.filter(~manifest.status.isin('uploaded_verified','already_verified')).count()==0
    from pyspark.sql import functions as F
    manifest_bytes=manifest.agg(F.sum('bytes')).first()[0]
    assert manifest_bytes==summary['verified_bytes'], (manifest_bytes,summary['verified_bytes'])
    manifest.selectExpr('entry.source AS source','status','bytes','seconds').groupBy('source','status').sum('bytes').show(truncate=False)
    put_json(CONFIG['buckets']['operations'],OP_PREFIX+'/_SUCCESS.json',summary)
    print('INGESTA_NACIONAL_COMPLETA_CONTRATO_1_0_0')
    print('Entrada de próximas notebooks:',summary['bundle_uri'])

## 12. Resultado y controles posteriores
Conserve run_id, PLAN_URI, bundle_uri, `_SUCCESS.json` y notebook ejecutada. Silver debe comprobar bundle y marcador. Los metadatos agrupados se leen como JSONL descomprimido; las páginas contienen los Items completos, geometrías, assets, fechas, baseline y metadatos de raster del distribuidor.

La revisión 4 cambia colección consultada, selección, empaquetado de metadatos y límite operativo. Se documenta en `config.ingestion_implementation`; no sobrescribe el project_config de 00. No cambia Perú, trimestres, cuatro bandas ni el umbral de cobertura. La colección contiene distintos baselines de procesamiento; Silver debe aplicar escalas/offsets conforme al Item/asset y verificar homogeneidad radiométrica antes de calcular índices o modelar. No se presenta la colección como baseline uniforme.

Sentinel Bronze no es un SAFE completo. INEI RAR permanece original, WorldPop es referencia 2020, OSIPTEL sin fecha observada no se convierte en historia. No se crean imágenes para completar faltantes ni se trasladan escenas de otros trimestres.

Referencias: [Earth Search](https://github.com/Element84/earth-search), [Ookla](https://github.com/teamookla/ookla-open-data), [Boto3](https://docs.aws.amazon.com/boto3/latest/guide/s3-uploading-files.html).